# Data Exploration

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import pm4py

In [ ]:
df = pm4py.read_xes('../data/bpic2017.xes')

## 1 Number of Cases
The number of cases in the dataset is determined by counting the unique values in the 'case:concept:name' column of the dataframe.

In [ ]:
case_count = df['case:concept:name'].nunique()
case_count

## 2. Number of Events
The number of events in the dataset is determined by counting the total number of rows in the dataframe (Event log)

In [ ]:
event_count = df.shape[0]
event_count

## 3. Number of Process Variants
A process variant is a unique sequence of activities that occur in the process. We can identify the number of process variants by grouping the events by case and then counting the unique sequences of activities.

In [ ]:
process_variants = pm4py.get_variants(df)
len(process_variants)

# 4. Number of Distinct Case & Event Attribute labels
Case attributes are properties associated with a case, while event attributes are properties associated with an event and can vary from event to event within the same case. In the dataset, case attributes are prefixed with "case:".

In [ ]:
case_attributes = [col for col in df.columns if col.startswith('case:')]
event_attributes = [col for col in df.columns if not col.startswith('case:')]

print(f"Number of distinct case attribute labels: {len(case_attributes)}")
print(f"Number of distinct event attribute labels: {len(event_attributes)}")

## Attribute ranges
Further analysis of the ranges of values for each attribute.

In [ ]:
id_cols = ['EventID', 'case:concept:name', 'OfferID']

numeric_ranges = []
for col in df.columns:
    s = df[col]
    if col in id_cols:
        continue
    if pd.api.types.is_datetime64_any_dtype(s) or (pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)):
        numeric_ranges.append({'attribute': col, 'min': s.min(), 'max': s.max(), 'missing': s.isna().sum()})

pd.DataFrame(numeric_ranges).set_index('attribute')

for col in df.columns:
    s = df[col]
    if col in id_cols or col in {r['attribute'] for r in numeric_ranges}:
        continue
    counts = s.value_counts()
    print(f"{col}: {len(counts)} variants, {s.isna().sum()} missing")
    for value, n in counts.items():
        print(f"    {value!r}: {n}")
    print()
    
pd.DataFrame(
    [{'attribute': col, 'distinct_values': df[col].nunique(), 'missing': df[col].isna().sum()} for col in id_cols]
).set_index('attribute')

## 5. Mean & Standard Deviation of Case Lengths
Case length = Number of events in a case

In [ ]:
case_event_counts = df.groupby('case:concept:name').size()
mean_case_event_count = case_event_counts.mean()
std_case_event_count = case_event_counts.std()

print(f"Mean number of events per case: {mean_case_event_count:.2f}")
print(f"Standard deviation of events per case: {std_case_event_count:.2f}")

## 6. Mean & Standard Deviation of Case Durations
Case Length = Total time taken for a case to complete: Measure the time difference between the first and last event of each case.

In [ ]:
case_start_end_times = df.groupby('case:concept:name')['time:timestamp'].agg(['min', 'max'])
case_start_end_times['case_length'] = (case_start_end_times['max'] - case_start_end_times['min']).dt.total_seconds()
mean_case_length = case_start_end_times['case_length'].mean()
std_case_length = case_start_end_times['case_length'].std()

def format_hours(total_seconds):
    total_seconds = int(total_seconds)
    days, rest = divmod(total_seconds, 86400)
    h, rest = divmod(rest, 3600)
    m, s = divmod(rest, 60)
    return f"{days:02d}d {h:02d}h {m:02d}m {s:02d}s"

print(f"Mean case length: {format_hours(mean_case_length)}")
print(f"Standard deviation of case length: {format_hours(std_case_length)}")

## 7. Addtional Statistics

### 7.1 Handovers per Case
A handover happens when two consecutive events of a case are executed by different resources. The number of handovers per case shows how often the work on a case changes hands.

In [ ]:
events = df.sort_values(['case:concept:name', 'time:timestamp'], kind='stable')
same_case = events['case:concept:name'] == events['case:concept:name'].shift()
resource_changed = events['org:resource'] != events['org:resource'].shift()
handovers_per_case = (same_case & resource_changed).groupby(events['case:concept:name']).sum()
resources_per_case = events.groupby('case:concept:name')['org:resource'].nunique()

print(f"Mean number of handovers per case: {handovers_per_case.mean():.2f}")
print(f"Standard deviation of handovers per case: {handovers_per_case.std():.2f}")
print(f"Median number of handovers per case: {handovers_per_case.median():.0f} (max: {handovers_per_case.max()})")
print(f"Mean number of different resources per case: {resources_per_case.mean():.2f}")
print(f"Cases without any handover: {(handovers_per_case == 0).mean():.2%}")

### 7.2 Offers per Case
A case can receive several loan offers. The number of offers per case is determined by counting the O_Create Offer events of each case.

In [ ]:
offers_per_case = (
    df[df['concept:name'] == 'O_Create Offer']
    .groupby('case:concept:name')
    .size()
    .reindex(df['case:concept:name'].unique(), fill_value=0)
)

print(f"Mean number of offers per case: {offers_per_case.mean():.2f}")
print(f"Standard deviation of offers per case: {offers_per_case.std():.2f}")
print("Distribution of offers per case:")
for n_offers, count in offers_per_case.value_counts().sort_index().items():
    print(f"    {n_offers} offers: {count} cases ({count / len(offers_per_case):.2%})")